# 01 · 전처리 & PDF 변환 (온도 시리즈)

**워크플로**
1. 폴더 불러오기 — 서브폴더 이름이 온도(예: `300K`), 각 폴더 안에 `.dm4`
2. 전처리 전/후 비교 — hot pixel 제거 · median · beam center
3. PDF 변환 과정 — I(q) → φ(q) → G(r)
4. 온도별 결과를 각각 `.npz`로 저장

재사용 함수는 모두 `fourdstem` 패키지에 들어 있습니다.
실제 `.dm4` 데이터가 없으면 `USE_SYNTHETIC=True`로 두면 데모용 합성 데이터로 전 셀이 실행됩니다.

In [ ]:
import os, glob
import numpy as np
import matplotlib.pyplot as plt
import fourdstem as fds

# ── 사용자 설정 ──────────────────────────────────────────────
DATA_ROOT = "/home/jonghoonk918/Desktop/fdstem/Amorphous/In-situ/Heating-SiO"        # 온도로 이름 붙은 .dm4 폴더 (예: 0025K.dm4 …)
OUT_DIR   = "/home/jonghoonk918/Desktop/fdstem/Amorphous/In-situ/Heating-SiOrdf_npz"  # 결과 npz 저장 폴더
USE_SYNTHETIC = not os.path.isdir(DATA_ROOT)   # 실데이터 없으면 자동 데모
if USE_SYNTHETIC:
    OUT_DIR = "rdf_npz"             # 데모/검증 실행용 로컬 폴더 (실데이터면 위 경로 사용)
N_JOBS = 4                          # 병렬 코어 수. 큰 4D 파일은 로딩 메모리 때문에 4~8 권장
LAZY   = False                      # 큰 4D는 to_pattern이 메모리-안전하게 평균. lazy(memmap)는 실험적.

# ⚑ q 단위 힌트: 일부 DM(dm4)은 검출기 역격자 단위를 '1/nm'로 잘못 저장합니다.
#   그 경우 "1/A"로 강제해서 잘못된 ÷10 을 막습니다. (단위가 정상이면 None)
Q_UNIT_HINT = "1/A"

# 중심빔(직접빔) 마스킹 — 이 반경(px) 안쪽은 제외. 회절판 보고 조절하세요.
#   보통 q_int_min / q_per_px 정도. (예: 0.15 / 0.0128 ≈ 12 px)
BEAM_RADIUS_PX = 12

# q 캘리브레이션: 첫 파일(최저온)의 G(r) 첫 피크를 이 값(Å)에 맞추고, 그 q_per_px를
#   전 온도에 고정. 온도에 따른 피크 이동은 신호이므로 다른 프레임은 1.61이 아닐 수 있음.
R_TARGET = 1.61                     # Si–O 기준 (필요시 조정)

# reduction 파라미터 — 이 패키지는 q = 1/d 컨벤션.
#   전자 PDF는 보통 max q(1/d) ~ 1.5 1/Å 근처 → 구간을 그에 맞춤 (0.8~12은 X!).
#   시리즈 전체에서 LOCK (온도 간 비교 가능하게).
# ★ 여기가 q_int_min·damping 입력 지점입니다 ★
#   q_int_min: FT 저-q 하한. 2c 표(FWHM/RMS)·그래프 보고 여기 값을 정한 뒤 처음부터 재실행하세요.
#     너무 높이면 첫 피크가 뭉개짐(FWHM 폭발). 보통 0.15~0.25. 직접빔은 중심빔 마스크가 처리합니다.
#   damping: 2d 비교 보고 "lorch"/"gauss"(+damping_b) 선택.
CFG = fds.RDFConfig(composition={"Si": 1, "O": 2},
                    q_int_min=0.20, q_int_max=1.50, r_min=1.10,
                    r_max=8.0, dr=0.02, damping="lorch", damping_b=0.4)
os.makedirs(OUT_DIR, exist_ok=True)
print("USE_SYNTHETIC =", USE_SYNTHETIC, "| N_JOBS =", N_JOBS, "| LAZY =", LAZY,
      "| cores =", os.cpu_count())

## 1) 폴더 불러오기

`fds.Series.from_directory(root)` 는 **레이아웃을 자동 감지**해서 온도 시리즈를 만듭니다:

- **평평한 파일** — 폴더 안에 온도로 이름 붙은 `.dm4` (예: `0025K.dm4 … 1100K.dm4`) →
  파일명에서 온도 파싱
- **서브폴더** — 온도별 서브폴더(예: `300K/scan.dm4`) → 폴더명에서 온도 파싱

`preprocess=` 훅으로 로드 즉시 hot/dead pixel 제거를 적용하고, `n_jobs`로 여러 파일을 **병렬 로드**,
`progress=True`로 **진행바**를 봅니다. `0025K` 처럼 앞에 0이 붙어도, `25K`/`300K`/`1100K` 모두 정확히 파싱됩니다.

**메모리 (큰 4D 파일)**: 파일 하나가 수 GB인 4D 큐브(예: 150×150×256×256 ≈ 5.8 GB)면, `lazy=True`로
**memmap 청크 평균**을 써서 파일을 통째로 RAM에 올리지 않습니다. 병렬 로딩은 동시에 여러 파일을 여니
`n_jobs`는 `4~8`처럼 보수적으로(코어 32개라도) 두는 게 안전합니다. RDF 계산 자체(작은 256×256)는
이후 `series.map`에서 코어를 많이 써도 됩니다.

In [ ]:
def make_ring(shape=(256, 256), r1=61.0, r2=110.0, hot=True, seed=0):
    '''데모용 비정질 링 패턴 (+ beam stop, hot pixel). r1(px)가 작을수록 G(r) 첫 피크가 큰 r.'''
    rng = np.random.default_rng(seed)
    H, W = shape
    yy, xx = np.mgrid[0:H, 0:W]
    cx, cy = W/2 + 3, H/2 - 2
    r = np.hypot(xx - cx, yy - cy)
    img = 4.0*np.exp(-r**2/(2*5**2))
    img += 1.0*np.exp(-(r-r1)**2/(2*14**2)) + 0.4*np.exp(-(r-r2)**2/(2*10**2))
    img += 0.03*rng.standard_normal(shape)
    img[:, W//2-1:W//2+1] = 0.0                     # beam stopper rod
    if hot:
        for _ in range(8):
            img[rng.integers(H), rng.integers(W)] += 3e3
    return np.clip(img, 0, None)

if USE_SYNTHETIC:
    temps = [300, 400, 500, 600, 700, 800, 900]
    # 온도↑ → 링 반경↓ (열팽창처럼 G(r) 첫 배위 r이 커지게).
    # 실데이터처럼 clean_pattern으로 hot pixel 제거한 패턴을 프레임에 저장.
    frames = [fds.Frame(pattern=fds.clean_pattern(make_ring(seed=T, r1=63 - 7*(T-300)/600)),
                        coord=float(T), label=f"{T}K", q_per_px=0.0125)
              for T in temps]
    series = fds.Series(frames)
else:
    # 평평한 파일(0025K.dm4 …)이든 서브폴더(300K/…)든 자동 감지.
    # q_unit_hint로 단위 강제, n_jobs로 병렬 로드, progress로 진행바.
    series = fds.Series.from_directory(DATA_ROOT, q_unit_hint=Q_UNIT_HINT,
                                       preprocess=fds.clean_pattern,
                                       n_jobs=N_JOBS, progress=True, lazy=LAZY)

print(f"{len(series)} frames:", series.labels())
print("temperatures:", series.coordinates())

# q 캘리브레이션 점검 — 데이터 q 범위가 변환 구간(CFG)과 겹치는지 확인
f0 = series[0]
q_per_px = f0.q_per_px or 1.0
q_max = q_per_px * min(f0.pattern.shape) / 2
print(f"\nq_per_px = {q_per_px:.4g} 1/Å/px,  data max q ≈ {q_max:.3g} 1/Å")
print(f"FT window = [{CFG.q_int_min}, {CFG.q_int_max}] 1/Å")
if q_max < CFG.q_int_max * 0.5:
    print("⚠️  data max q 가 변환 구간보다 훨씬 작습니다 — Q_UNIT_HINT='1/A' 필요하거나 "
          "CFG 구간을 낮추세요 (q=1/d 컨벤션).")

## 2) 전처리 전/후 — hot pixel · median · beam center

한 프레임을 골라 **원본 → hot pixel 제거 → beam stop 마스크 + Friedel 중심**을 나란히 봅니다.
(`from_folders`에 `preprocess`를 넣으면 이 정리는 이미 적용되어 있으니, 여기선 비교용으로 원본을 다시 만듭니다.)

In [ ]:
frame = series[0]
raw = make_ring(seed=int(frame.coord)) if USE_SYNTHETIC else frame.pattern
q_per_px = frame.q_per_px or 1.0

# 전처리
cleaned, hot_mask = fds.remove_hot_pixels(raw, threshold=8, return_mask=True)
cleaned = fds.remove_dead_pixels(cleaned)
stopper = fds.beam_stopper_mask(cleaned)
(cx, cy), fried = fds.find_center(cleaned, stopper)
print(f"hot pixels removed: {hot_mask.sum()},  center=({cx:.1f},{cy:.1f}),  Friedel={fried:.2f}")

fig, ax = plt.subplots(1, 3, figsize=(14, 4.4))
fds.show_pattern(raw, ax=ax[0], title="raw")
fds.show_pattern(cleaned, ax=ax[1], title="hot/dead removed")
beam = fds.disk_mask(cleaned.shape, (cx, cy), BEAM_RADIUS_PX)   # 중심빔(직접빔) 원판
full_mask = fds.combine_masks(stopper, beam)                    # stopper + 중심빔
fds.show_pattern(cleaned, center=(cx, cy), mask=full_mask, ax=ax[2],
                 title="masks (stopper + center-beam) + center")
print(f"masked px: stopper+beam = {int(full_mask.sum())}  (beam r = {BEAM_RADIUS_PX}px)")
fig.tight_layout(); plt.show()

## 2b) q 캘리브레이션 — 첫 파일로 고정

**첫 파일(최저온)** 의 G(r) 첫 피크를 `R_TARGET`(=1.61 Å)에 맞도록 `q_per_px`를 보정하고,
그 값을 **모든 온도에 그대로 고정**합니다. G(r) 피크 위치는 `q_per_px`에 반비례하므로
`q_per_px_new = q_per_px × (관측 r / R_TARGET)`. 온도에 따른 피크 이동은 **신호**이니 다른 프레임은
1.61이 아닐 수 있습니다(그게 정상).

In [ ]:
ref = series[0]                                    # 첫 파일 = 최저온 (정렬됨)
_stop = fds.beam_stopper_mask(ref.pattern)
(_cx, _cy), _ = fds.find_center(ref.pattern, _stop)

def _first_peak(qpp):
    rr = fds.pattern_to_rdf(ref.pattern, qpp, CFG, center=(_cx, _cy),
                            center_beam_radius=BEAM_RADIUS_PX)
    r1, _ = fds.first_peak_position(rr.r, rr.Gr, 1.3, 2.6)
    return r1, rr

Q_RAW = ref.q_per_px or 1.0
r_before, res_before = _first_peak(Q_RAW)

# 반복 보정 — 피크가 R_TARGET에 수렴할 때까지 (q_per_px∝1/r 이 정확치 않으므로)
Q_PER_PX = Q_RAW
for _ in range(6):
    r_i, _ = _first_peak(Q_PER_PX)
    if abs(r_i - R_TARGET) < 0.003:
        break
    Q_PER_PX *= r_i / R_TARGET
r_after, res_after = _first_peak(Q_PER_PX)

print(f"[calib] ref = {ref.label}:  first peak {r_before:.3f} Å (raw) -> {r_after:.3f} Å (target {R_TARGET})")
print(f"[calib] q_per_px: {Q_RAW:.5g}  ->  {Q_PER_PX:.5g} 1/Å/px   (locked for ALL frames)")

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(res_before.r, res_before.Gr, label=f"before  (peak {r_before:.2f} Å)")
ax.plot(res_after.r,  res_after.Gr,  label=f"after   (peak {r_after:.2f} Å)")
ax.axvline(R_TARGET, color="0.6", ls="--"); ax.set_xlim(0, 4); ax.axhline(0, color="0.85", lw=0.8)
ax.set_xlabel("r (Å)"); ax.set_ylabel("G(r)"); ax.set_title(f"{ref.label}: q calibration to {R_TARGET} Å")
ax.legend(); plt.show()

## 2c) q_int_min 고르기 — 정량 지표로 객관적 선택

세 열로 판단합니다:
- **FWHM** (가장 중요): 첫 피크 폭. q_min을 너무 키우면 저-q 정보를 잃어 **피크가 뭉개지며 FWHM이
  폭발**합니다(예: 28 Å = 파괴). **작고 안정적이어야** 좋음.
- **RMS/peak**: 저-r 평탄도를 **피크 높이로 정규화**(정규화 안 하면 q_min↑일 때 진폭이 줄어 RMS도 줄어
  착시). 낮을수록 좋지만 **FWHM 폭발과 함께면 무의미**.
- **FSDP=ok**: MRO 보존 (q_min ≤ 0.6 × q_FSDP). φ의 FSDP(빨간 선)를 침범하면 CLIP.

> 중심빔 마스크가 이미 q < "마스크 q-하한"을 제거하므로 그보다 낮은 q_min은 효과 동일(직접빔은 마스크가
> 처리). **→ FWHM이 작고 G(r)가 안정적인 가장 낮은 q_min**(보통 0.15~0.25)을 고르세요.
> 고른 값은 **맨 위 config의 `CFG.q_int_min`에 넣고 처음부터 재실행** → 캘리브레이션도 그 q_min으로
> 되어 25K 첫 피크가 정확히 1.61이 됩니다(첫 피크는 q_min에 약하게 의존).

In [ ]:
fm = series[len(series)//2]; pat = fm.pattern
_st = fds.beam_stopper_mask(pat); (cx, cy), _ = fds.find_center(pat, _st)

# FSDP 위치 (calibrated q) — 낮은 q_min의 φ에서 첫 양의 피크
_cfg0 = fds.RDFConfig(composition=CFG.composition, q_int_min=0.12,
                      q_int_max=CFG.q_int_max, r_min=CFG.r_min, r_max=CFG.r_max,
                      dr=CFG.dr, damping=CFG.damping)
_r0 = fds.pattern_to_rdf(pat, Q_PER_PX, _cfg0, center=(cx, cy),
                         center_beam_radius=BEAM_RADIUS_PX)
q_fsdp, _ = fds.first_peak_position(_r0.q_reduced, _r0.phi, 0.30, 1.20)
q_floor = BEAM_RADIUS_PX * Q_PER_PX          # 마스크가 정하는 실질 하한
print(f"FSDP ≈ {q_fsdp:.3f} 1/Å  |  마스크 q-하한 ≈ {q_floor:.3f}  |  q_int_min 권장 ≤ {0.6*q_fsdp:.2f}\n")
# RMS/peak: 피크높이로 정규화(진폭 축소 편향 제거).  FWHM: 폭발하면 피크 파괴.
print(f"{'q_min':>6} {'RMS/peak':>9} {'1st r(Å)':>9} {'FWHM(Å)':>8}   FSDP")

fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
for qmin in [0.10, 0.15, 0.20, 0.25, 0.35, 0.45]:
    cfg = fds.RDFConfig(composition=CFG.composition, q_int_min=qmin,
                        q_int_max=CFG.q_int_max, r_min=CFG.r_min,
                        r_max=CFG.r_max, dr=CFG.dr, damping=CFG.damping)
    rr = fds.pattern_to_rdf(pat, Q_PER_PX, cfg, center=(cx, cy),   # calibrated q 고정
                            center_beam_radius=BEAM_RADIUS_PX)
    r1, h1 = fds.first_peak_position(rr.r, rr.Gr, 1.3, 2.2)
    fit = fds.fit_gaussian_peak(rr.r, rr.Gr, 1.3, 2.0)
    rms = rr.diagnostics.get("sub_rmin_rms", float("nan"))
    rel = rms / abs(h1) if h1 else float("nan")         # 정규화된 저-r 평탄도
    fwhm = fit["fwhm"]
    tag = "ok" if qmin <= 0.6*q_fsdp else "CLIP!"
    if fwhm > 2.0:
        tag += "  <peak destroyed>"
    print(f"{qmin:>6.2f} {rel:>9.3f} {r1:>9.3f} {fwhm:>8.2f}   {tag}")
    ax[0].plot(rr.q_reduced, rr.phi, lw=1, label=f"{qmin}")
    ax[1].plot(rr.r, rr.Gr, lw=1, label=f"{qmin}")
ax[0].axvline(q_fsdp, color="red", ls="--", lw=1.3, label=f"FSDP {q_fsdp:.2f}")
ax[0].axhline(0, color="0.7", lw=0.8); ax[0].set_xlim(0, 1.5)
ax[0].set_xlabel("q (1/Å)"); ax[0].set_ylabel("φ(q)")
ax[0].set_title("φ(q): keep q_min left of FSDP"); ax[0].legend(fontsize=7, title="q_min")
ax[1].axhline(0, color="0.7", lw=0.8); ax[1].set_xlim(0, 4)
ax[1].set_xlabel("r (Å)"); ax[1].set_ylabel("G(r)")
ax[1].set_title("G(r): sharp stable peak = good (watch FWHM)"); ax[1].legend(fontsize=7, title="q_min")
plt.tight_layout(); plt.show()

## 2d) damping 비교 — ripple 억제 (신호와 겹침 완화)

저-r 잔물결(termination ripple)은 낮은 Qmax 탓이라 q_min으로는 못 없앱니다(오히려 q_min↑ 이면 커짐).
**감쇠(damping)** 로 ripple 진폭 자체를 눌러 신호와 덜 겹치게 합니다. `gauss`가 `lorch`보다 더 강하게
누르지만 피크가 약간 넓어집니다. 셋(none/lorch/gauss)을 겹쳐 비교해 고르세요.

In [ ]:
fm = series[len(series)//2]; pat = fm.pattern
_st = fds.beam_stopper_mask(pat); (cx, cy), _ = fds.find_center(pat, _st)
fig, ax = plt.subplots(figsize=(8, 4.5))
for damp, kw in [("none", {}), ("lorch", {}), ("gauss", {"damping_b": 0.4})]:
    cfg = fds.RDFConfig(composition=CFG.composition, q_int_min=CFG.q_int_min,
                        q_int_max=CFG.q_int_max, r_min=CFG.r_min, r_max=CFG.r_max,
                        dr=CFG.dr, damping=damp, **kw)
    rr = fds.pattern_to_rdf(pat, Q_PER_PX, cfg, center=(cx, cy),
                            center_beam_radius=BEAM_RADIUS_PX)
    lab = damp + (f" (b={kw['damping_b']})" if kw else "")
    ax.plot(rr.r, rr.Gr, lw=1.2, label=lab)
ax.axhline(0, color="0.8", lw=0.8); ax.set_xlim(0, 6)
ax.set_xlabel("r (Å)"); ax.set_ylabel("G(r)")
ax.set_title("damping comparison (ripple suppression)"); ax.legend()
plt.show()

## 3) 온도별 처리 (병렬 + 진행바) & npz 저장

각 온도의 평균 패턴을 **중심빔(직접빔) + beam stopper 제외**하고 PDF 변환합니다
(`center_beam_radius=BEAM_RADIUS_PX`). 중심·스케일 N만 프레임별로 잡고 나머지 reduction 파라미터는
LOCK. `series.map(..., n_jobs, progress)`로 병렬·진행바, 저장은 메인에서 순서대로.
`fds.save_rdf`가 `q, Iq, φ, r, Gr`를 함께 저장 → 02 노트북에서 그대로 다시 불러옵니다.

In [ ]:
# 프레임별 RDF (부작용 없는 순수 함수) — 중심빔+stopper 제외, 캘리브레이션 고정(Q_PER_PX)
def process_frame(f):
    pat = f.pattern
    stop = fds.beam_stopper_mask(pat)
    (cx, cy), _ = fds.find_center(pat, stop)               # center는 프레임별
    rr = fds.pattern_to_rdf(pat, Q_PER_PX, CFG, center=(cx, cy),   # q_per_px는 고정
                            center_beam_radius=BEAM_RADIUS_PX)
    return dict(coord=f.coord, label=f.label, result=rr)

results = series.map(process_frame, n_jobs=(1 if USE_SYNTHETIC else N_JOBS),
                     progress=True, desc="RDF")
results.sort(key=lambda o: o["coord"])

summary = []
for out in results:
    rr = out["result"]
    path = os.path.join(OUT_DIR, f"{int(out['coord'])}K_rdf.npz")
    fds.save_rdf(path, rr, temperature=out["coord"], source=out["label"])
    r1, _ = fds.first_peak_position(rr.r, rr.Gr, 1.3, 2.2)
    summary.append((out["coord"], r1))
    print(f"  T={out['coord']:>6.0f}K  N={rr.N:.3g}  r1={r1:.3f} Å  →  {os.path.basename(path)}")
print(f"\n저장 완료: {len(summary)} 개 npz → {OUT_DIR}/")

## 3b) RDF QC — 환원이 잘 됐는가 (자동 판정)

곡선 피팅이 아니라 스케일 N 하나로 저-r을 평평하게 만드는 환원이라, 품질 = **① 첫 피크가 기대 위치
(~1.61 Å), ② φ(q) 고-q 발산 없음, ③ 저-r에 가짜 봉우리 없음**. `fds.rdf_quality`가 온도별로 자동 판정합니다.
- `1st r`/`off`: 첫 피크 위치와 목표(1.61)로부터 편차
- `lowr_rms`: 저-r 평탄도(낮을수록 좋음)   · `phi_drift`: 고-q φ 드리프트(|값|<1.5면 램프 없음)
- `qmax`: 사용된 최대 q(=1/d) — 실공간 분해능 한계. verdict가 `check`면 그 온도를 확인.

In [ ]:
print(f"{'T(K)':>6} {'1st r':>6} {'off':>7} {'lowr_rms':>9} {'phi_drift':>9} {'qmax':>5}  verdict")
n_good = 0
for out in results:
    Q = fds.rdf_quality(out["result"], expected_first_peak=R_TARGET)
    n_good += (Q["verdict"] == "good")
    print(f"{out['coord']:>6.0f} {Q['first_peak_r']:>6.3f} {Q['first_peak_offset']:>+7.3f} "
          f"{Q['low_r_rms']:>9.3g} {Q['phi_highq_drift']:>9.2f} {Q['qmax']:>5.2f}  {Q['verdict']}")
print(f"\n{n_good}/{len(results)} good.  "
      f"참고: qmax~1.6(1/d)라 FWHM 넓음 → 상대 변화는 신뢰, 절대 정량(배위수)은 제한적.")

## 4) 파일마다 변형 과정 — 모든 온도의 I(q) → φ(q) → G(r)

각 온도(파일)가 **어떻게 변형되는지 전부** 한 그림에 겹쳐 봅니다(무지개색, 온도순). 세 단계
(I(q) → φ(q) → G(r))별로 온도 변화를 직접 비교할 수 있습니다.

In [ ]:
temps = np.array([o["coord"] for o in results])
cm = plt.get_cmap("rainbow")
norm = plt.Normalize(np.nanmin(temps), np.nanmax(temps))

fig, ax = plt.subplots(1, 3, figsize=(15, 4.6))
for o in results:
    rr = o["result"]; c = cm(norm(o["coord"]))
    ax[0].plot(rr.q, rr.Iq, color=c, lw=1)
    ax[1].plot(rr.q_reduced, rr.phi, color=c, lw=1)
    ax[2].plot(rr.r, rr.Gr, color=c, lw=1)
ax[0].set_title("(1) I(q)");  ax[0].set_xlabel("q (1/Å)"); ax[0].set_ylabel("I(q)")
ax[1].set_title("(2) φ(q)");  ax[1].set_xlabel("q (1/Å)"); ax[1].set_ylabel("φ(q)"); ax[1].axhline(0, color="0.7", lw=0.8)
ax[2].set_title("(3) G(r)");  ax[2].set_xlabel("r (Å)");   ax[2].set_ylabel("G(r)"); ax[2].axhline(0, color="0.7", lw=0.8)
sm = plt.cm.ScalarMappable(cmap=cm, norm=norm); sm.set_array([])
fig.colorbar(sm, ax=ax, label="temperature (K)", fraction=0.03, pad=0.01)
plt.show()

## 5) 무지개 워터폴 (겹치지 않게 위로 쌓기)

같은 결과를 **offset으로 쌓아** 곡선이 겹치지 않게 봅니다.

In [ ]:
temps = np.array([o["coord"] for o in results])
fig, ax = plt.subplots(1, 2, figsize=(13, 7))
fds.plot_series_waterfall([(o["result"].q, o["result"].Iq) for o in results], temps,
                          ax=ax[0], cmap="rainbow", xlabel="q (1/Å)",
                          labels=[f"{int(T)}K" for T in temps])
ax[0].set_title("I(q) vs temperature (rainbow waterfall)")
fds.plot_series_waterfall([(o["result"].r, o["result"].Gr) for o in results], temps,
                          ax=ax[1], cmap="rainbow", xlabel="r (Å)")
ax[1].set_title("G(r) vs temperature (rainbow waterfall)")
plt.tight_layout(); plt.show()

## 6) QC — 스케일 N & 중심 vs 온도 (빔 커런트/두께 드리프트 감시)

환원의 스케일 **N은 (빔 커런트 × 두께 × dose)** 를 흡수합니다. 빔 커런트가 온도에 따라 떨어지면 N도
같이 떨어지지만 **φ(q)·G(r) 모양(peak 위치)은 영향받지 않습니다** — N이 그 배율을 통째로 먹기 때문.
그래서 N을 온도의 함수로 보면 **부드러우면 정상**, **급점프면 포화/초점/두께 이벤트**를 의심할 수 있습니다.
(절대 진폭을 온도끼리 직접 비교하지 마세요. 위치·상대 변화만 신뢰.)

In [ ]:
temps = np.array([o["coord"] for o in results])
Ns    = np.array([o["result"].N for o in results])
frieds = np.array([o["result"].diagnostics.get("friedel_corr", np.nan) for o in results])
cents = np.array([o["result"].center for o in results])   # (n, 2) beam center 이동

fig, ax = plt.subplots(1, 3, figsize=(15, 4))
ax[0].plot(temps, Ns, "o-", color="tab:purple")
ax[0].set_xlabel("temperature (K)"); ax[0].set_ylabel("scale N")
ax[0].set_title("scale N vs T  (beam-current / thickness drift)")
ax[1].plot(temps, frieds, "o-", color="tab:green")
ax[1].set_xlabel("temperature (K)"); ax[1].set_ylabel("Friedel corr")
ax[1].set_title("center quality (closer to 1 is better)"); ax[1].set_ylim(0, 1.02)
ax[2].plot(temps, cents[:, 0], "o-", label="cx"); ax[2].plot(temps, cents[:, 1], "s-", label="cy")
ax[2].set_xlabel("temperature (K)"); ax[2].set_ylabel("beam center (px)")
ax[2].set_title("beam center drift"); ax[2].legend()
plt.tight_layout(); plt.show()

다음: **02 노트북**에서 이 `npz`들을 불러와 PCA 분해·차분맵·2nd shell/FSDP로 온도 의존성을 분석합니다.